# 零、总述

本章我们来介绍多模态，如下图所示，如果一个语言模型可以像人类那样看一张图片，听一段语音等等，则此模型会被称为**多模态模型**。他不仅可以是**输入多模态**，也可以是**输出多模态**，也就是大模型可以生成图片，视频，语音等等内容

<center>
<img src="./resources/multimodal.png">
</center>

我们通常称**多模态(大)模型**为**Multimodal Large Language Model**, 而不是**Multimodal Large Model**，*这里要注意的是⚠️⚠️⚠️: 这里多加了 `language` 并没有翻译错*，而是因为 **<span style="color: red">Language（语言）绝不仅仅是一个孤立的词语。（英文原文：Language does not solely live in a vacuum）</span>**。

举个例子来说，你的*肢体语言、面部表情、说话的语调等等*都是交流的方式，它会让别人更加理解你在说什么。所以，**多模态模型**是 LLM 的一个扩展。

我们接下来讲一下**多模态模型的架构**，目前最广泛使用的架构为**Modality Encoder -> Projecter/Adapter -> LLM -> Output**，我们将这四个步骤分成了四层, 每一层的详细解释如下表所示

| **层级**      | **核心功能**                                  | **代表技术**                   | **输入 -> 输出**             |
|-------------|-------------------------------------------|----------------------------|--------------------------|
| 模态编码层       | 提取不同模态的信息                                 | ViT、CLIP、SigLIP、Whisper等   | 图像/音频/视频 -> 特征向量         |
| 模态对齐层       | 将不同模态的特征转换为 LLM 可有效利用的表示 | Linear、MLP Projecter、Q-Former等 | 模态特征向量 -> 对齐后的特征向量（多模态表示） |
| 多模态融合层 | 建立图像、文本等模态之间的关联，并进行理解和推理                  | Transformer、Self-Attention等 | 多模态表示 -> 隐藏状态     |
| 输出生成层       | 根据融合的结果生成目标模态                         | LM Head、Diffusion、Audio Decoder 等 | 隐藏状态 -> 文本/图像/音频/视频 |

① **模态编码层（Modality Encoding）**

**负责将原始信息转换成神经网络可以处理的特征表示**，可以使用 ViT, Video Encoder 等技术来编码多模态信息。CLIP 和 SigLIP 是一种预训练的方法或者是模型，Whisper 是语音识别的模型。

② **模态对齐层（Modality Alignment）**

它的核心目标是：**将视觉、音频等不同模态的特征转换为 LLM 能够有效处理和理解的表示。** 它主要有三个核心的功能：**维度对齐、语义适配和特征压缩**，这三个功能在模态对齐层不一定全部存在。其中，

- **维度对齐**

它的作用是**将视觉等模态特征映射到 LLM 所需的隐藏维度**，一般使用 Linear Projection 或 MLP Projector 来实现，Linear Projection 的底层是通过线性变换，而 MLP Projector 的底层是通过多层线性变换和非线性激活学习，更灵活的特征映射。

- **语义适配**

**它让映射后的特征能够被 LLM 有效利用，建立视觉信息与语言语义的关联**，一般使用 Q-former、经过多模态训练的 Projector 等来实现。

- **特征压缩**

**它主要是提取重要的模态信息，减少输入 LLM 的 token数量**，实现的技术一般是 Q-former 和 Resampler 等

③ **多模态融合层（Multimodal Fusion）**

模型对齐层解决的是 **不同模态的特征如何接入 LLM**，那么**多模态融合层**解决的就是**LLM 如何综合利用这些特征。**
这一层的核心作用是：**让不同模态的特征相互交互，建立跨模态关联，并结合上下文完成理解和推理**，它主要基于 Transformer 架构实现

④ **输出生成层（Output Generation）**

它负责将融合后的特征或条件信息转换为目标输出。常见的实现技术有 LM Head、Diffusion Model、Audio Decoder 和 Video Decoder/Generator 等。注意：**Diffusion Model** 不仅仅是一个简单的解码器，而是一套生成模型。


下面是详细的架构图

<center>
<img src="./resources/multimodal_architecture.png">
</center>

本章主要介绍视觉领域的一些技术，从 ViT 开始，随后讲解不同模态的编码器，接下来讲解如何让 LLM 可以处理多模态信息，最后介绍多模态生成的相关技术

# 一、视觉的 Transformer

我们可以将 Transformer 扩展到计算机视觉领域，这称之为 **ViT(Vision Transformer) 架构**，它在识别图像的任务上比之前的CNN 网络表现的性能更好。与 Transformer 类似，ViT 会将非结构化的数据（图片）转换为数据表示，以用于各种各样的任务当中，比如分类。如下图所示

<center>
<img src="./resources/Transformer_and_ViT.png">
</center>

ViT 使用了 Transformer 架构中比较重要的组件：**编码器**，它会将一段文本的输入转换为数字的表示，之后传到**解码器**中。然而，在编码器工作之前，文本的输入首先**需要分词（tokenize）**，即将一整个句子分成多个 token 的序列。因为图像不是一个词的序列，所以我们不能对其直接进行分词，但我们可以模拟分词这个思想，将图像分成每个“词”。

想象你有一张 512 x 512 像素的猫的图像，每一个单个的像素不代表什么，但是，你如果将这些像素进行分块（**patches**），它就能代表很多信息。ViT 使用了这种方式，**它把原始的图像转成了图像的分块（patches of images）**,也就是说，它将图片水平和竖直的切分成了块（patches）。如下图所示：

<center>
<img src="./resources/patched_image.png">
</center>

虽然，我们将其切成了块，这类似于文本中的词，但是，我们不能给这些块（patches）分配具体的 ID，因为在其他图片中，我们几乎很少能找见与之对应的块（patches）。那么，我们可以借助 embedding 模型来创建对应的数字表示，它可以作为 Transformer 模型的输入。这种方法（如下图所示）最有趣的地方在于，当这些嵌入传递到编码器之后，编码器会将它们视为文本的 token，从这一点来看，训练文本和图像没有任何区别。由于这种相似性，ViT 可以使得很多语言模型变得多模态，使用 ViT 最直接的方式就体现在训练嵌入模型的过程中。

<center>
<img src="./resources/ViT.png">
</center>

# 二、多模态编码与嵌入技术

在之前的章节中，我们使用了 Embedding 模型来捕捉文本表示的语义关系，比如论文或者文档的嵌入。在 LLM 的应用中，Embedding 是最重要的**引擎或者说是驱动者**，对于从大量的信息中提取某个信息，这种方式是最有效的，**现在我们可以将文本和视觉的表示都嵌入到同一个向量空间**中。如下图所示

<center>
<img src="./resources/multimodal_embedding_model.png">
</center>

这样的话，我们就可以基于文本来搜索图像了。例如：如果我们搜到了“小狗的图片”这段文字，那么它和哪个图像最接近呢？反过来，通过图片搜文字也是类似的操作。下图展示了嵌入空间

<center>
<img src="./resources/embedding_vector_space.png">
</center>

多模态的嵌入模型有很多，最有名的模型是**CLIP（Contrastive Language-Image Pre-training）**，与它类似的还有 *SigLIP/SigLIP2, DINOv2/DINOv3*，**CLIP、SigLIP/SigLIP2** 重点研究的图像与文本之间的语义对齐，**DINOv2/DINOv3** 重点研究如何从图像本身学习高质量的视觉特征，不依赖图文配对进行核心视觉预训练

## 2.1 CLIP(Contrastive Language-Image Pre-training)

### 2.1.1 基本介绍

CLIP 是一个嵌入模型，采用的是双编码器架构（Dual Encoder），它能计算图像和文本的嵌入，最终的嵌入向量依赖于相同的向量空间，它意味着图像可以和文本作比较，此模型可用于如下任务：

- Zero-shot classification
- Clustering
- Search
- Generation

### 2.1.2 CLIP 如何生成多模态的 Embedding

想象你有上百万张图片和对应的标题，那么这个数据集就能创建两个表示：图像和标题。

为了实现这个功能，CLIP 首先用文本编码器去嵌入文本并且用图像编码器去嵌入图像，这是第一步，如下图所示：

<center>
<img src="./resources/CLIP_First_Step.png">
</center>

接下来，CLIP 的第二步是去优化图文向量之间的相似度，我们使用图文相似度矩阵来展示，其训练目标是：**正确的图文对相似度尽可能高，错误的图文对相似度尽可能低**，如下图所示

<center>
<img src="./resources/CLIP_Second_Step.png">
</center>

在计算它们的相似度之后，模型的权重会被更新并且它会依赖于新一批的数据去更新表示，此方法叫做 Contrastive Learning，如下图所示

<center>
<img src="./resources/CLIP_Third_Step.png">
</center>

CLIP 使用 Softmax 作为它的损失函数，其公式如下：

$$
Loss_{CLIP} = \frac{1}{2} (Loss_{I \rightarrow T} + Loss_{I \rightarrow T})
$$

其中：

- $Loss_{I \rightarrow T}$ 代表 Image-To-Text 的损失，即沿着**每一行**计算 Softmax
- $Loss_{T \rightarrow I}$ 代表 Text-To-Image 的损失，即沿着**每一列**计算 Softmax

有一点需要说明的是，上述的两个方向是都需要计算的，因为**图像检索文本和文本检索图像是两个不同的条件分布**，即使它们都在同一个相似度矩阵中，但行方向和列方向的归一化分母不同，产生的损失和梯度也不同

### 2.1.3 OpenCLIP

## 2.2 SigLIP/SigLIP2

**SigLIP 和 SigLIP2 都是 Google 提出的视觉-语言编码模型（Vision-Language Encoder）**,主要用于学习图像与文本之间的语义关系，让图像和文本能够映射到可比较的特征空间。

两者的核心区别是：

- **SigLIP(2023)**：在 CLIP 双编码器架构的基础上，使用 Sigmoid Loss 替代 Softmax 对比损失，优化图文对齐训练
- **SigLIP(2025)**: 保留 SigLIP 的核心架构，增加了自监督学习、图像描述训练等技术，强化多语言理解、局部定位和密集视觉特征

### 2.2.1 SigLIP：基于 Sigmoid Loss 的图文对齐模型

SigLIP 全称是 Sigmoid Loss for Language-Image Pre-Training，它与 CLIP 的总体架构相似，采用的都是双编码器架构。它与 CLIP 的主要区别在于损失函数。

CLIP 使用的是 Softmax 对比损失，它会在一个 Batch 内构造图文相似度矩阵，通过 Softmax 让正确配对获得更高的相对概率。

SigLIP 使用的是 Sigmoid Loss, 它**将每个图文配对视为一个独立的二分类判断**

$$
P(匹配 | I, T) = \sigma(score(I,T))
$$

其中：

- $\sigma(·)$ 表示 Sigmoid 函数
- $score(I, T)$ 是图像和文本 Embedding 的相似度经过缩放、偏置处理后的得分

训练时，正样本的匹配概率趋近于 1， 负样本趋近于 0

### 2.2.2 SigLIP2：增强语义理解和空间感知

SigLIP2 的完整论文名称为：Multilingual Vision-Language Encoder with Improved Semantic Understanding, Localization, and Dense Features(2025 年 2 月发布)

SigLIP2 的主要升级是**通过多种训练目标，让视觉编码器不仅理解整张图片的语义，还能更好地保留局部区域的空间信息**

SigLIP2 的四项关键改进如下：

① **多语言理解（Multilingual Understanding）**

使用更丰富的多语言数据与训练策略，让图像能够更好地与中文、英文等不同语言的描述进行语义匹配，而不只是侧重英文

② **局部定位（Localization）**

通过加入带有位置监督的图像描述训练，让模型更好地理解物体和图像区域的关系。例如：模型不只知道图片中有一只猫，还能学习“左下角的猫”对应哪个区域

③ **密集视觉特征（Dense Features）**

强化各个图像 Patch 的特征表达，使它们包含更多局部语义和空间信息。这些特征更适合下游语义分割、深度估计等需要像素级或区域级理解的任务

④ **灵活分辨率（NaFlex）**

引入能够处理不同分辨率、保留原始宽高比的 NaFlex 版本，减少固定比例缩放带来的信息损失，对文档，长图等输入尤其有价值

## 2.3 DINOv2/DINOv3

# 三、让文本生成模型支持多模态

## 3.1 BLIP-2：Bridging the Modality Gap

## 3.2 预处理多模态输入

## 3.3 使用示例：图像描述

## 3.4 使用示例：基于聊天的多模态提示词

# 四、多模态内容的生成